<a href="https://colab.research.google.com/github/rahul238xaviers/mathsLearning/blob/main/nemo_guardRail_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Tools
First, we install the libraries needed to run the AI, the guardrails, and the chat screen.

In [14]:
# Install the updated toolkit for Gemini 2.5 Flash
!pip install -q -U \
    langchain-google-genai \
    langgraph \
    nemoguardrails \
    gradio

# Step 2: Setup Keys and Folders
Create a secret in the colab. You can click on keys where you can define the secret name as "GOOGLE_API_KEY" and set the value which you have. If you want to use OPEN_API_KEY then you can use that as well. Please remember to update the model name, library or chat wrapper when you update the OPEN_API_KEY

In [20]:
import os
import gradio as gr
from typing import List, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from google.colab import userdata
from langgraph.graph import StateGraph, END
from nemoguardrails import RailsConfig, LLMRails

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
MODEL_ID = "models/gemini-2.5-flash" # Note: Use 1.5 or 2.0 based on current stable availability

os.makedirs("config", exist_ok=True)

# Step 3: Define the Rules (Guardrails)
We tell the AI to stay on the topic of flights and refuse everything else.

In [21]:
# 1. Update config.yml with 'google_genai'
config_yaml = f"""
models:
  - type: main
    engine: google_genai
    model: {MODEL_ID}
"""

# 2. Update rails.co
rails_co = """
define user ask off_topic
  "What is the weather?"
  "Tell me a joke"
  "Who is the president?"

define bot refuse off_topic
  "I only provide flight fares."

define flow off_topic
  user ask off_topic
  bot refuse off_topic
"""

with open("config/config.yml", "w") as f:
    f.write(config_yaml)

with open("config/rails.co", "w") as f:
    f.write(rails_co)

# 3. Initialize Rails
config = RailsConfig.from_path("./config")
rails = LLMRails(config)

# Step 4: Create the Brain (LangGraph)
We create two steps:
1. Input Guard: Checks if the question is allowed.
2. Flight Agent: If allowed, calculates the fare.

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from typing import List, TypedDict

class State(TypedDict):
    messages: List[BaseMessage]
    allowed: bool
    rejection: str

async def check_input(state: State):
    """
    Node 1: Input Guard. Uses NeMo Rails to decide if we proceed.
    """
    user_text = state['messages'][-1].content

    # Get the official response from NeMo Guardrails
    bot_res = await rails.generate_async(prompt=user_text)

    # We check if NeMo triggered any refusal.
    # Instead of an exact string match, we check for core keywords
    # defined in your rails.co 'bot refuse' intent.
    refusal_keywords = ["only help with flight", "only provide flight", "specialized assistant"]

    is_blocked = any(keyword in bot_res for keyword in refusal_keywords)

    if is_blocked:
        return {"allowed": False, "rejection": bot_res}

    return {"allowed": True}

def get_fare(state: State):
    """
    Node 2: The Agent. Only executes if Node 1 gave the green light.
    """
    # If Node 1 blocked it, we return the NeMo refusal message and stop.
    if not state.get("allowed", True):
        return {"messages": [AIMessage(content=state.get("rejection", "I can only help with flight fares."))]}

    # Initialize Gemini 2.5 Flash
    llm = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0)

    # Strict System Instruction to prevent the agent from 'hallucinating'
    # that a joke is a fare request.
    system_instructions = (
        "You are a strict flight fare assistant. "
        "If the user asks for anything other than a flight price (like a joke or general info), "
        "respond: 'I am sorry, I can only provide flight fare details.'"
    )

    # We pass the system instructions + the actual user message
    res = llm.invoke([HumanMessage(content=system_instructions)] + state["messages"])
    return {"messages": [res]}

# Build the Graph
builder = StateGraph(State)

builder.add_node("guard", check_input)
builder.add_node("agent", get_fare)

builder.set_entry_point("guard")
builder.add_edge("guard", "agent")
builder.add_edge("agent", END)

bot_app = builder.compile()

# Step 5: Launch Chat
Run this cell to start the chat window inside Colab.

In [ ]:
async def chat(msg, history):
    try:
        # Run the LangGraph app
        result = await bot_app.ainvoke({"messages": [HumanMessage(content=msg)]})

        # Check if the guardrails set 'allowed' to False
        is_blocked = not result.get("allowed", True)

        final_msg = result["messages"][-1].content

        if is_blocked:
            return f"🚫 [GUARDRAIL BLOCK]: {final_msg}"

        return final_msg

    except Exception as e:
        # This catches code crashes so you don't get a silent red 'Error' box
        return f"⚠️ [SYSTEM ERROR]: {str(e)}"

# Set debug=True to see full stack traces in the Colab output
gr.ChatInterface(
    chat,
    title="Gemini 2.5 Flight Agent (Guarded)",
    description="I only discuss flight fares. Try 'Tell me a joke' to see a block."
).launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e7ea08fb04a27b2934.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
